# 4장 장기 메모리와 검색 증강 생성

이전 대화를 저장하고 다시 사용하는 방법을 살펴봅니다.

[Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/)와 [Jay Alammar](https://www.linkedin.com/in/jalammar)의 도서 [An Illustrated Guide to AI Agents](https://www.oreilly.com/library/view/an-illustrated-guide/9798341662681/)에 수록된 4장 실습입니다. [전체 실습 목차](../README.md) · [원본 저장소](https://github.com/HandsOnLLM/An-Illustrated-Guide-To-AI-Agents)

### Google Colab에서 패키지 설치하기 (선택)

Colab 등 클라우드 노트북에서는 아래 설치 셀의 주석을 해제하고 실행합니다. 원격 모델 API를 사용한다면 Ollama 설치 셀은 건너뜁니다.

In [ ]:
# %%capture
# !pip install illustrated-agents

Colab 안에서 Ollama로 모델을 실행하려면 **런타임 → 런타임 유형 변경 → 하드웨어 가속기 → T4 GPU**를 선택한 뒤, 아래 셀의 주석을 해제하고 실행합니다.

In [ ]:
# !apt-get install -y zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
# !nohup ollama serve > /dev/null 2>&1 & sleep 3 && ollama pull gemma3:12b && ollama pull embeddinggemma &

---

## 1 - 언어 모델 선택

이번 실습에 사용할 언어 모델을 지정합니다.

In [ ]:
from illustrated_agents.chapters.ch2 import LLM

# Gemma 3 12B(내장 추론과 도구 호출 없음)
llm = LLM(model="gemma3:12b")

다른 언어 모델을 사용하려면 아래에서 로컬 서버 또는 클라우드 API에 해당하는 설정을 선택합니다.

In [ ]:
# # llama.cpp
# llm = LLM(model="gemma-3-12B-it-Q4_K_M", base_url="http://127.0.0.1:8080")

# # LM Studio
# llm = LLM(model="gemma-3-12b-it", base_url="http://127.0.0.1:1234/v1")

# # OpenRouter
# import os
# llm = LLM(model="google/gemma-3-12b-it", base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"])

## 2. 검색 증강 생성(RAG) 추가

앞에서는 대화를 잘라 내거나 요약하는 단기 메모리를 다뤘습니다. 이번에는 외부에 보관한 문서에서 관련 내용을 검색해 답변에 활용하는 장기 메모리를 구현합니다.

먼저 OpenAI 호환 엔드포인트를 호출하는 `EmbeddingModel` 클래스를 정의합니다. [EmbeddingGemma](https://ai.google.dev/gemma/docs/embeddinggemma)는 텍스트를 벡터로 바꾸는 3억 800만 매개변수 모델입니다. 앞서 답변을 생성할 때 쓴 120억 매개변수 모델보다 작습니다.

In [ ]:
import json
import urllib.request

class EmbeddingModel:
    """임베딩을 생성한다."""

    def __init__(
        self,
        model: str,
        base_url: str = "http://localhost:11434/v1",
    ):
        """지정한 임베딩 모델로 클라이언트를 초기화한다."""
        self.model = model
        self.base_url = base_url

    def embed(self, text: str) -> list[float]:
        """텍스트를 수치 벡터로 변환한다."""
        # OpenAI 호환 /embeddings 엔드포인트로 POST 요청
        request = urllib.request.Request(
            f"{self.base_url}/embeddings",
            data=json.dumps({"model": self.model, "input": text}).encode(),
            headers={"Content-Type": "application/json"},
        )
        with urllib.request.urlopen(request) as resp:
            response = json.loads(resp.read())

        # 임베딩을 추출해 반환
        return response["data"][0]["embedding"]

Ollama에서 임베딩 모델을 실행하므로 기존 클라이언트를 사용해 클래스를 초기화할 수 있습니다.

In [ ]:
# EmbeddingGemma 초기화
embedding_model = EmbeddingModel(model="embeddinggemma")

# 임베딩 모델 시험
output = embedding_model.embed("Dolphins are amazing!")
output

예제의 출력은 -1과 1 사이의 값 768개로 구성된 벡터입니다.

두 문서 벡터의 유사도는 코사인 유사도로 비교할 수 있습니다. 두 벡터의 내적을 두 벡터 길이의 곱으로 나눈 값입니다. 벡터 사이의 각도가 작을수록 코사인 유사도가 높습니다. 아래 예제로 계산해 봅니다.

In [ ]:
from rich import print

# 임베딩 생성
embedding_a = embedding_model.embed("I love flamingos.")
embedding_b = embedding_model.embed("Dolphins use echolocation.")
embedding_c = embedding_model.embed("Flamingos are pink birds.")

# A와 B의 코사인 유사도 계산
dot_ab = sum(x * y for x, y in zip(embedding_a, embedding_b))
norm_a = sum(x * x for x in embedding_a) ** 0.5
norm_b = sum(x * x for x in embedding_b) ** 0.5
similarity_ab = dot_ab / (norm_a * norm_b)

# A와 C의 코사인 유사도 계산
dot_ac = sum(x * y for x, y in zip(embedding_a, embedding_c))
norm_c = sum(x * x for x in embedding_c) ** 0.5
similarity_ac = dot_ac / (norm_a * norm_c)

print(f"Similarity between [bold]'I love flamingos.'[/bold] and [bold]'Dolphins use echolocation.'[/bold]: {similarity_ab}")
print(f"Similarity between [bold]'I love flamingos.'[/bold] and [bold]'Flamingos are pink birds.'[/bold]: {similarity_ac}")

`RAGMemory`에서 검색하고 문서를 전달하는 순서는 다음과 같습니다.

1. 초기화할 때 외부 문서를 각각 임베딩합니다.
2. `embedding_model.embed(query)`로 사용자 질문도 임베딩합니다.
3. 질문 벡터와 각 문서 벡터의 유사도를 계산합니다.
4. 유사도가 높은 문서를 선택합니다.
5. 선택한 문서를 프롬프트에 추가합니다.

In [ ]:
from illustrated_agents.chapters.ch4 import Memory

class RAGMemory(Memory):
    """검색 증강 생성으로 장기 메모리를 구현한다."""

    def __init__(self, embedding_model: EmbeddingModel, documents: list[str]):
        super().__init__()
        self.embedding_model = embedding_model
        self.documents = documents
        self.embeddings = [embedding_model.embed(doc) for doc in documents]

    def add(self, role: str, content: str, **kwargs) -> None:
        # 사용자 질문에 검색 결과를 추가한 뒤 저장
        if role == "user":
            context = "\n".join(self.search(content))
            content = f"""Context:
{context}

Question: {content}"""
        super().add(role, content, **kwargs)

    def search(self, query: str) -> list[str]:
        """질문과 유사도가 높은 문서 k개를 반환한다."""
        query_embed = self.embedding_model.embed(query)
        scores = [self._cosine(query_embed, embed) for embed in self.embeddings]
        ranked = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
        return [self.documents[index] for index in ranked[:3]]

    def _cosine(self, a: list[float], b: list[float]) -> float:
        """두 임베딩 간의 코사인 유사도를 계산한다."""
        dot = sum(x * y for x, y in zip(a, b))
        norm = (sum(x * x for x in a) ** 0.5) * (sum(x * x for x in b) ** 0.5)
        return dot / norm

에이전트가 직접 볼 수 없는 문서를 몇 개 준비합니다. 문서가 수천 개일 때에도 질문과 관련된 문서만 골라 모델에 보내는 원리는 같습니다.

문서에는 Sarah와 Ilse의 직업, 거주지, 취미와 좋아하는 동물이 적혀 있습니다. 질문은 “Sarah가 가장 좋아하는 동물은 무엇인가요?”입니다. 검색 결과에 Sarah가 홍학을 좋아한다는 문장이 포함되는지 봅니다.

In [ ]:
from illustrated_agents.chapters.ch4 import TinyAgent

# 외부 문서를 사용하는 RAGMemory
documents = [
    "Sarah works as a marine biologist studying coral reefs.",
    "Sarah lives in Lisbon, Portugal.",
    "Sarah's favorite hobby is rock climbing.",
    "Sarah favorite animals are flamingos.",
    "Sarah speaks fluent Spanish and Portuguese.",
    "Ilse is a software engineer at a renewable energy startup.",
    "Ilse lives in Amsterdam, the Netherlands.",
    "Ilse plays the cello in a local string quartet.",
    "Ilse's favorite author is Brandon Sanderson.",
    "Ilse's favorite animals are dolphins.",
]
memory = RAGMemory(documents=documents, embedding_model=embedding_model)

# 에이전트 생성 후 질문 실행
agent = TinyAgent(llm=llm, memory=memory)
response = agent.run("What is Sarah's favorite animal?")
print(response)

In [ ]:
print(agent.memory.get_messages())

예제에서는 유사도 순으로 세 문서를 가져옵니다. 모델에 보내는 입력은 작아지지만, 세 문서 모두 질문과 관련이 낮을 수도 있습니다. 관련성이 낮은 문서를 제외하려면 유사도 최솟값을 정하고 그 기준을 넘는 문서만 선택할 수 있습니다.

---

# 구현한 내용

대화 기록을 관리하는 `Memory`를 `TinyAgent`에 연결했습니다. 아래에서 언어 모델, 메모리, 에이전트의 구성을 확인합니다.

In [ ]:
from illustrated_agents.chapters.ch4 import what_we_built_lt; what_we_built_lt

# 다음 장

[5장](../chapter05/chapter05.ipynb)에서는 에이전트의 도구 호출을 한 단계씩 구현합니다.